In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, trim, lower, when, lit, current_timestamp,
    to_json, struct
)
from pyspark.sql.window import Window
import uuid

# =========================================================
# CONFIG
# =========================================================
pipeline_name = "retail_pipeline"
layer_name = "silver"
load_type = "INCREMENTAL"
triggered_by = "manual"
source_system = "S3-Retail-Lakehouse-Bucket"

audit_table = "retail.audit.etl_run_log"
reject_table = "retail.audit.etl_reject_log"

batch_id = f"batch_silver_incremental_{str(uuid.uuid4())[:8]}"

In [0]:
def log_layer_processed_run(
    source_run_id,
    target_run_id,
    table_name,
    status,
    rows_read,
    rows_written,
    rows_rejected,
    start_time,
    end_time,
    error_message=None
):
    safe_error = "NULL" if error_message is None else "'" + error_message.replace("'", "''")[:5000] + "'"

    spark.sql(f"""
        INSERT INTO retail.audit.layer_processed_runs
        SELECT
          uuid() AS tracking_id,
          '{source_run_id}' AS source_run_id,
          '{target_run_id}' AS target_run_id,
          '{pipeline_name}' AS pipeline_name,
          'bronze' AS source_layer,
          'silver' AS target_layer,
          '{table_name}' AS table_name,
          '{status}' AS status,
          {rows_read} AS rows_read,
          {rows_written} AS rows_written,
          {rows_rejected} AS rows_rejected,
          timestamp('{start_time}') AS start_time,
          timestamp('{end_time}') AS end_time,
          unix_timestamp(timestamp('{end_time}')) - unix_timestamp(timestamp('{start_time}')) AS duration_seconds,
          {safe_error} AS error_message,
          current_timestamp() AS created_at,
          current_timestamp() AS updated_at
    """)

In [0]:
# =========================================================
# HELPERS
# =========================================================
def generate_run_id():
    return str(uuid.uuid4())


def start_audit(run_id, table_name):
    spark.sql(f"""
        INSERT INTO {audit_table}
        (
          run_id, batch_id, pipeline_name, layer_name, table_name, source_system,
          load_type, triggered_by, start_time, end_time, duration_seconds,
          status, rows_read, rows_written, rows_rejected, records_before,
          records_after, error_message, created_at
        )
        VALUES (
          '{run_id}', '{batch_id}', '{pipeline_name}', '{layer_name}', '{table_name}', '{source_system}',
          '{load_type}', '{triggered_by}', current_timestamp(), NULL, NULL,
          'STARTED', NULL, NULL, NULL, NULL,
          NULL, NULL, current_timestamp()
        )
    """)


def end_audit_success(run_id, rows_read, rows_written, rows_rejected, records_before, records_after):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'SUCCESS',
          rows_read = {rows_read},
          rows_written = {rows_written},
          rows_rejected = {rows_rejected},
          records_before = {records_before},
          records_after = {records_after}
        WHERE run_id = '{run_id}'
    """)


def end_audit_fail(run_id, error_message):
    safe_error = error_message.replace("'", "''")[:5000]
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'FAILED',
          error_message = '{safe_error}'
        WHERE run_id = '{run_id}'
    """)


def log_rejects(df_rejects, run_id, table_name):
    if df_rejects.isEmpty():
        return

    (
        df_rejects
        .withColumn("reject_id", F.expr("uuid()"))
        .withColumn("run_id", lit(run_id))
        .withColumn("batch_id", lit(batch_id))
        .withColumn("pipeline_name", lit(pipeline_name))
        .withColumn("layer_name", lit(layer_name))
        .withColumn("table_name", lit(table_name))
        .withColumn("source_system", lit(source_system))
        .withColumn("rejected_at", current_timestamp())
        .withColumn("created_at", current_timestamp())
        .select(
            "reject_id", "run_id", "batch_id", "pipeline_name", "layer_name",
            "table_name", "source_system", "source_record_id", "reject_stage",
            "reject_reason", "reject_column", "severity", "raw_payload",
            "error_code", "error_message", "rejected_at", "created_at"
        )
        .write.format("delta").mode("append").saveAsTable(reject_table)
    )


def normalize_string(c):
    return when(trim(col(c)) == "", None).otherwise(trim(col(c)))


def build_typed_expr(source_col, rule):
    raw = normalize_string(source_col)

    if rule == "string":
        return raw
    if rule == "int":
        return raw.cast("int")
    if rule == "int_nullable":
        return raw.cast("int")
    if rule == "decimal":
        return raw.cast("decimal(18,2)")
    if rule == "timestamp":
        return raw.cast("timestamp")
    if rule == "date":
        return raw.cast("date")
    if rule == "date_nullable":
        return raw.cast("date")
    if rule == "boolean_flag":
        return (
            when(lower(raw).isin("true", "1", "yes"), F.lit(True))
            .when(lower(raw).isin("false", "0", "no"), F.lit(False))
            .otherwise(F.lit(False))
        )

    return raw


def build_invalid_condition(df, typed_columns, required_columns):
    condition = F.lit(False)

    for c in required_columns:
        condition = condition | col(c).isNull() | (trim(col(c)) == "")

    for c, rule in typed_columns:
        if rule == "int":
            condition = condition | (
                normalize_string(c).isNotNull() &
                build_typed_expr(c, "int").isNull()
            )
        elif rule == "decimal":
            condition = condition | (
                normalize_string(c).isNotNull() &
                build_typed_expr(c, "decimal").isNull()
            )
        elif rule == "timestamp":
            condition = condition | (
                normalize_string(c).isNotNull() &
                build_typed_expr(c, "timestamp").isNull()
            )
        elif rule == "date":
            condition = condition | (
                normalize_string(c).isNotNull() &
                build_typed_expr(c, "date").isNull()
            )
        elif rule == "boolean_flag":
            condition = condition | (
                normalize_string(c).isNotNull() &
                (~lower(normalize_string(c)).isin("true", "1", "yes", "false", "0", "no"))
            )

    return condition

# Silver Incremental MERGE Processor

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, trim, lower, when, current_timestamp, to_json, struct
from pyspark.sql.window import Window
import uuid
from datetime import datetime

# =========================================================
# CONFIG
# =========================================================
pipeline_name = "retail_pipeline"
layer_name = "silver"
load_type = "INCREMENTAL"
triggered_by = "manual"
source_system = "S3-Retail-Lakehouse-Bucket"

audit_table = "retail.audit.etl_run_log"
reject_table = "retail.audit.etl_reject_log"
processed_runs_table = "retail.audit.layer_processed_runs"

batch_id = f"batch_silver_incremental_{str(uuid.uuid4())[:8]}"


#==========================================================
# Configuration
#==========================================================
table_configs = [
    {
        "table_name": "ns_brands",
        "source_table": "retail.bronze.ns_brands",
        "target_table": "retail.silver.ns_brands",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("brand_code", "string"),
            ("brand_name", "string"),
            ("is_active", "boolean_flag"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": ["internal_id", "brand_code", "brand_name", "created_at"],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_customers",
        "source_table": "retail.bronze.ns_customers",
        "target_table": "retail.silver.ns_customers",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("entity_id", "string"),
            ("company_name", "string"),
            ("customer_type", "string"),
            ("email", "string"),
            ("phone", "string"),
            ("subsidiary", "string"),
            ("currency_code", "string"),
            ("terms_name", "string"),
            ("vat_number", "string"),
            ("billing_address1", "string"),
            ("billing_city", "string"),
            ("billing_state", "string"),
            ("billing_country", "string"),
            ("shipping_address1", "string"),
            ("shipping_city", "string"),
            ("shipping_state", "string"),
            ("shipping_country", "string"),
            ("is_active", "boolean_flag"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "entity_id", "company_name", "customer_type",
            "subsidiary", "currency_code", "terms_name", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_items",
        "source_table": "retail.bronze.ns_items",
        "target_table": "retail.silver.ns_items",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("item_id", "string"),
            ("item_name", "string"),
            ("item_type", "string"),
            ("category", "string"),
            ("subcategory", "string"),
            ("brand_internal_id", "int_nullable"),
            ("base_price", "decimal"),
            ("cost_price", "decimal"),
            ("is_active", "boolean_flag"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "item_id", "item_name", "item_type",
            "category", "subcategory", "base_price", "cost_price", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_sales_orders",
        "source_table": "retail.bronze.ns_sales_orders",
        "target_table": "retail.silver.ns_sales_orders",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("tran_id", "string"),
            ("customer_internal_id", "int"),
            ("order_date", "date"),
            ("order_status", "string"),
            ("sales_channel", "string"),
            ("payment_status", "string"),
            ("currency_code", "string"),
            ("external_ref_num", "string"),
            ("integration_source", "string"),
            ("order_total", "decimal"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "tran_id", "customer_internal_id", "order_date",
            "order_status", "sales_channel", "payment_status", "currency_code",
            "integration_source", "order_total", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_sales_order_lines",
        "source_table": "retail.bronze.ns_sales_order_lines",
        "target_table": "retail.silver.ns_sales_order_lines",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("sales_order_internal_id", "int"),
            ("line_num", "int"),
            ("item_internal_id", "int"),
            ("quantity", "int"),
            ("unit_rate", "decimal"),
            ("line_amount", "decimal"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "sales_order_internal_id", "line_num",
            "item_internal_id", "quantity", "unit_rate", "line_amount", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_shipments",
        "source_table": "retail.bronze.ns_shipments",
        "target_table": "retail.silver.ns_shipments",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("shipment_number", "string"),
            ("sales_order_internal_id", "int"),
            ("warehouse_name", "string"),
            ("shipment_date", "date"),
            ("delivery_date", "date_nullable"),
            ("shipment_status", "string"),
            ("tracking_number", "string"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "shipment_number", "sales_order_internal_id",
            "warehouse_name", "shipment_date", "shipment_status", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    }
]

# =========================================================
# AUDIT HELPERS
# =========================================================

def generate_run_id():
    return str(uuid.uuid4())


def start_audit(run_id, table_name):
    spark.sql(f"""
        INSERT INTO {audit_table}
        (
          run_id, batch_id, pipeline_name, layer_name, table_name, source_system,
          load_type, triggered_by, start_time, end_time, duration_seconds,
          status, rows_read, rows_written, rows_rejected, records_before,
          records_after, error_message, created_at
        )
        VALUES (
          '{run_id}', '{batch_id}', '{pipeline_name}', '{layer_name}', '{table_name}', '{source_system}',
          '{load_type}', '{triggered_by}', current_timestamp(), NULL, NULL,
          'STARTED', NULL, NULL, NULL, NULL,
          NULL, NULL, current_timestamp()
        )
    """)


def end_audit_success(run_id, rows_read, rows_written, rows_rejected, records_before, records_after):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'SUCCESS',
          rows_read = {rows_read},
          rows_written = {rows_written},
          rows_rejected = {rows_rejected},
          records_before = {records_before},
          records_after = {records_after}
        WHERE run_id = '{run_id}'
    """)


def end_audit_fail(run_id, error_message):
    safe_error = error_message.replace("'", "''")[:5000]

    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'FAILED',
          error_message = '{safe_error}'
        WHERE run_id = '{run_id}'
    """)


def log_rejects(df_rejects, run_id, table_name):
    if df_rejects.isEmpty():
        return

    df_rejects.write.format("delta").mode("append").saveAsTable(reject_table)

# =========================================================
# LAYER RUN TRACKING
# =========================================================
def get_unprocessed_bronze_run_ids(table_name):
    return [
        r.run_id
        for r in spark.sql(f"""
            SELECT DISTINCT f.run_id
            FROM retail.audit.ingested_files f
            LEFT ANTI JOIN {processed_runs_table} p
              ON f.run_id = p.source_run_id
             AND f.table_name = p.table_name
             AND p.source_layer = 'bronze'
             AND p.target_layer = 'silver'
             AND p.status = 'SUCCESS'
            WHERE f.table_name = '{table_name}'
              AND f.ingestion_status = 'SUCCESS'
        """).collect()
    ]


def log_layer_processed_run(
    source_run_id,
    target_run_id,
    table_name,
    status,
    rows_read,
    rows_written,
    rows_rejected,
    start_time,
    end_time,
    error_message=None
):
    safe_error = "NULL" if error_message is None else "'" + error_message.replace("'", "''")[:5000] + "'"

    spark.sql(f"""
        INSERT INTO {processed_runs_table}
        SELECT
          uuid() AS tracking_id,
          '{source_run_id}' AS source_run_id,
          '{target_run_id}' AS target_run_id,
          '{pipeline_name}' AS pipeline_name,
          'bronze' AS source_layer,
          'silver' AS target_layer,
          '{table_name}' AS table_name,
          '{status}' AS status,
          CAST({rows_read} AS BIGINT) AS rows_read,
          CAST({rows_written} AS BIGINT) AS rows_written,
          CAST({rows_rejected} AS BIGINT) AS rows_rejected,
          timestamp('{start_time}') AS start_time,
          timestamp('{end_time}') AS end_time,
          unix_timestamp(timestamp('{end_time}')) - unix_timestamp(timestamp('{start_time}')) AS duration_seconds,
          {safe_error} AS error_message,
          current_timestamp() AS created_at,
          current_timestamp() AS updated_at
    """)

# =========================================================
# SILVER HELPER FUNCTIONS
# =========================================================

def normalize_string(c):
    return trim(col(c).cast("string"))


def build_typed_expr(c, rule):
    value = trim(col(c).cast("string"))

    if rule == "string":
        return value

    elif rule == "int":
        return value.cast("int")

    elif rule == "int_nullable":
        return when(value == "", None).otherwise(value.cast("int"))

    elif rule == "decimal":
        return value.cast("decimal(18,2)")

    elif rule == "timestamp":
        return value.cast("timestamp")

    elif rule == "date":
        return value.cast("date")

    elif rule == "date_nullable":
        return when(value == "", None).otherwise(value.cast("date"))

    elif rule == "boolean_flag":
        return (
            when(lower(value).isin("true", "1", "yes", "y"), lit(True))
            .when(lower(value).isin("false", "0", "no", "n"), lit(False))
            .otherwise(None)
        )

    else:
        return col(c)


def build_invalid_condition(df, typed_columns, required_columns):
    invalid_condition = None

    # 1. Required column null/blank check
    for c in required_columns:
        cond = col(c).isNull() | (trim(col(c).cast("string")) == "")
        invalid_condition = cond if invalid_condition is None else (invalid_condition | cond)

    # 2. Type cast validation
    for c, rule in typed_columns:
        raw_value = trim(col(c).cast("string"))

        if rule in ["int", "int_nullable"]:
            casted_value = raw_value.cast("int")

            cond = (
                raw_value.isNotNull()
                & (raw_value != "")
                & casted_value.isNull()
            )

        elif rule == "decimal":
            casted_value = raw_value.cast("decimal(18,2)")

            cond = (
                raw_value.isNotNull()
                & (raw_value != "")
                & casted_value.isNull()
            )

        elif rule == "timestamp":
            casted_value = raw_value.cast("timestamp")

            cond = (
                raw_value.isNotNull()
                & (raw_value != "")
                & casted_value.isNull()
            )

        elif rule in ["date", "date_nullable"]:
            casted_value = raw_value.cast("date")

            cond = (
                raw_value.isNotNull()
                & (raw_value != "")
                & casted_value.isNull()
            )

        elif rule == "boolean_flag":
            cond = (
                raw_value.isNotNull()
                & (raw_value != "")
                & (~lower(raw_value).isin("true", "false", "1", "0", "yes", "no", "y", "n"))
            )

        else:
            cond = lit(False)

        invalid_condition = cond if invalid_condition is None else (invalid_condition | cond)

    return invalid_condition

# =========================================================
# MAIN SILVER INCREMENTAL MERGE
# =========================================================
def process_table_incremental_merge(cfg):
    table_name = cfg["table_name"]
    source_table = cfg["source_table"]
    target_table = cfg["target_table"]
    business_key = cfg["business_key"]
    typed_columns = cfg["typed_columns"]
    required_columns = cfg["required_columns"]

    run_id = generate_run_id()
    start_time = datetime.now()

    start_audit(run_id, table_name)

    try:
        records_before = spark.table(target_table).count()

        bronze_run_ids = get_unprocessed_bronze_run_ids(table_name)

        if not bronze_run_ids:
            end_time = datetime.now()

            end_audit_success(
                run_id=run_id,
                rows_read=0,
                rows_written=0,
                rows_rejected=0,
                records_before=records_before,
                records_after=records_before
            )

            print(f"{target_table}: no new Bronze runs to process")
            return

        print(f"{table_name}: processing Bronze run_ids = {bronze_run_ids}")

        # Read only new Bronze runs
        df_bronze = (
            spark.table(source_table)
            .filter(col("etl_run_id").isin(bronze_run_ids))
        )

        rows_read = df_bronze.count()

        if rows_read == 0:
            end_time = datetime.now()

            end_audit_success(
                run_id=run_id,
                rows_read=0,
                rows_written=0,
                rows_rejected=0,
                records_before=records_before,
                records_after=records_before
            )

            print(f"{target_table}: Bronze run IDs found, but no matching rows in Bronze")
            return

        # Validate records
        invalid_condition = build_invalid_condition(
            df_bronze,
            typed_columns,
            required_columns
        )

        df_rejects = (
            df_bronze
            .filter(invalid_condition)
            .withColumn("source_record_id", normalize_string(business_key))
            .withColumn("reject_stage", lit("SILVER"))
            .withColumn("reject_reason", lit("INVALID_CAST_OR_REQUIRED_FIELD"))
            .withColumn("reject_column", lit(",".join(required_columns)))
            .withColumn("severity", lit("ERROR"))
            .withColumn("raw_payload", to_json(struct(*[col(c) for c in df_bronze.columns])))
            .withColumn("error_code", lit("SLV_001"))
            .withColumn("error_message", lit("Required field missing/blank or failed type cast"))
        )

        rows_rejected = df_rejects.count()
        log_rejects(df_rejects, run_id, table_name)

        df_valid = df_bronze.filter(~invalid_condition)

        # Type casting
        typed_select_exprs = [
            build_typed_expr(c, rule).alias(c)
            for c, rule in typed_columns
        ]

        df_silver = df_valid.select(*typed_select_exprs)

        # Deduplicate within incremental batch
        window_spec = Window.partitionBy(business_key).orderBy(
            col("etl_loaded_at").desc(),
            col("record_hash").desc()
        )

        df_silver_deduped = (
            df_silver
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(col("rn") == 1)
            .drop("rn")
        )

        target_cols = spark.table(target_table).columns
        df_silver_deduped = df_silver_deduped.select(*target_cols)

        rows_prepared_for_merge = df_silver_deduped.count()

        df_silver_deduped.createOrReplaceTempView(
            f"vw_{table_name}_silver_incremental"
        )

        update_set = ",\n".join([
            f"target.{c} = source.{c}"
            for c in target_cols
            if c != business_key
        ])

        insert_cols = ", ".join(target_cols)
        insert_vals = ", ".join([f"source.{c}" for c in target_cols])

        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING vw_{table_name}_silver_incremental AS source
            ON target.{business_key} = source.{business_key}

            WHEN MATCHED AND target.record_hash <> source.record_hash THEN
              UPDATE SET
                {update_set}

            WHEN NOT MATCHED THEN
              INSERT ({insert_cols})
              VALUES ({insert_vals})
        """

        spark.sql(merge_sql)

        records_after = spark.table(target_table).count()
        end_time = datetime.now()

        end_audit_success(
            run_id=run_id,
            rows_read=rows_read,
            rows_written=rows_prepared_for_merge,
            rows_rejected=rows_rejected,
            records_before=records_before,
            records_after=records_after
        )

        # Mark Bronze runs as processed into Silver
        for bronze_run_id in bronze_run_ids:
            run_df = df_bronze.filter(col("etl_run_id") == bronze_run_id)

            run_rows_read = run_df.count()

            run_reject_condition = build_invalid_condition(
                run_df,
                typed_columns,
                required_columns
            )

            run_rows_rejected = run_df.filter(run_reject_condition).count()

            run_valid_df = run_df.filter(~run_reject_condition)

            run_rows_written = run_valid_df.select(business_key).distinct().count()

            log_layer_processed_run(
                source_run_id=bronze_run_id,
                target_run_id=run_id,
                table_name=table_name,
                status="SUCCESS",
                rows_read=run_rows_read,
                rows_written=run_rows_written,
                rows_rejected=run_rows_rejected,
                start_time=start_time,
                end_time=end_time
            )

        print(f"{target_table}: Silver incremental MERGE completed")
        print(f"Rows read: {rows_read}")
        print(f"Rows rejected: {rows_rejected}")
        print(f"Rows prepared for merge: {rows_prepared_for_merge}")
        print(f"Before: {records_before}")
        print(f"After: {records_after}")

    except Exception as e:
        end_time = datetime.now()
        error_message = str(e)

        end_audit_fail(run_id, error_message)

        print(f"{target_table}: FAILED")
        print(error_message)

        raise


for cfg in table_configs:
    process_table_incremental_merge(cfg)

## New Incremetal follow SCD2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, trim, lower, when, current_timestamp, to_json, struct, expr
from pyspark.sql.window import Window
import uuid
from datetime import datetime

# =========================================================
# CONFIG
# =========================================================
pipeline_name = "retail_pipeline"
layer_name = "silver"
load_type = "INCREMENTAL"
triggered_by = "manual"
source_system = "S3-Retail-Lakehouse-Bucket"

audit_table = "retail.audit.etl_run_log"
reject_table = "retail.audit.etl_reject_log"
processed_runs_table = "retail.audit.layer_processed_runs"

batch_id = f"batch_silver_incremental_{str(uuid.uuid4())[:8]}"


#==========================================================
# Configuration
#==========================================================
table_configs = [
    {
        "table_name": "ns_brands",
        "source_table": "retail.bronze.ns_brands",
        "target_table": "retail.silver.ns_brands",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("brand_code", "string"),
            ("brand_name", "string"),
            ("is_active", "boolean_flag"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": ["internal_id", "brand_code", "brand_name", "created_at"],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_customers",
        "source_table": "retail.bronze.ns_customers",
        "target_table": "retail.silver.ns_customers",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("entity_id", "string"),
            ("company_name", "string"),
            ("customer_type", "string"),
            ("email", "string"),
            ("phone", "string"),
            ("subsidiary", "string"),
            ("currency_code", "string"),
            ("terms_name", "string"),
            ("vat_number", "string"),
            ("billing_address1", "string"),
            ("billing_city", "string"),
            ("billing_state", "string"),
            ("billing_country", "string"),
            ("shipping_address1", "string"),
            ("shipping_city", "string"),
            ("shipping_state", "string"),
            ("shipping_country", "string"),
            ("is_active", "boolean_flag"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "entity_id", "company_name", "customer_type",
            "subsidiary", "currency_code", "terms_name", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_items",
        "source_table": "retail.bronze.ns_items",
        "target_table": "retail.silver.ns_items",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("item_id", "string"),
            ("item_name", "string"),
            ("item_type", "string"),
            ("category", "string"),
            ("subcategory", "string"),
            ("brand_internal_id", "int_nullable"),
            ("base_price", "decimal"),
            ("cost_price", "decimal"),
            ("is_active", "boolean_flag"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "item_id", "item_name", "item_type",
            "category", "subcategory", "base_price", "cost_price", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_sales_orders",
        "source_table": "retail.bronze.ns_sales_orders",
        "target_table": "retail.silver.ns_sales_orders",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("tran_id", "string"),
            ("customer_internal_id", "int"),
            ("order_date", "date"),
            ("order_status", "string"),
            ("sales_channel", "string"),
            ("payment_status", "string"),
            ("currency_code", "string"),
            ("external_ref_num", "string"),
            ("integration_source", "string"),
            ("order_total", "decimal"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "tran_id", "customer_internal_id", "order_date",
            "order_status", "sales_channel", "payment_status", "currency_code",
            "integration_source", "order_total", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_sales_order_lines",
        "source_table": "retail.bronze.ns_sales_order_lines",
        "target_table": "retail.silver.ns_sales_order_lines",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("sales_order_internal_id", "int"),
            ("line_num", "int"),
            ("item_internal_id", "int"),
            ("quantity", "int"),
            ("unit_rate", "decimal"),
            ("line_amount", "decimal"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "sales_order_internal_id", "line_num",
            "item_internal_id", "quantity", "unit_rate", "line_amount", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    },
    {
        "table_name": "ns_shipments",
        "source_table": "retail.bronze.ns_shipments",
        "target_table": "retail.silver.ns_shipments",
        "business_key": "internal_id",
        "typed_columns": [
            ("internal_id", "int"),
            ("shipment_number", "string"),
            ("sales_order_internal_id", "int"),
            ("warehouse_name", "string"),
            ("shipment_date", "date"),
            ("delivery_date", "date_nullable"),
            ("shipment_status", "string"),
            ("tracking_number", "string"),
            ("created_at", "timestamp"),
            ("etl_loaded_at", "timestamp"),
            ("source_system", "string"),
            ("etl_run_id", "string"),
            ("ingestion_date", "date"),
            ("record_hash", "string"),
            ("is_deleted", "boolean_flag")
        ],
        "required_columns": [
            "internal_id", "shipment_number", "sales_order_internal_id",
            "warehouse_name", "shipment_date", "shipment_status", "created_at"
        ],
        "dedupe_keys": ["internal_id", "record_hash"]
    }
]

# =========================================================
# AUDIT HELPERS
# =========================================================

def generate_run_id():
    return str(uuid.uuid4())


def start_audit(run_id, table_name):
    spark.sql(f"""
        INSERT INTO {audit_table}
        (
          run_id, batch_id, pipeline_name, layer_name, table_name, source_system,
          load_type, triggered_by, start_time, end_time, duration_seconds,
          status, rows_read, rows_written, rows_rejected, records_before,
          records_after, error_message, created_at
        )
        VALUES (
          '{run_id}', '{batch_id}', '{pipeline_name}', '{layer_name}', '{table_name}', '{source_system}',
          '{load_type}', '{triggered_by}', current_timestamp(), NULL, NULL,
          'STARTED', NULL, NULL, NULL, NULL,
          NULL, NULL, current_timestamp()
        )
    """)


def end_audit_success(run_id, rows_read, rows_written, rows_rejected, records_before, records_after):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'SUCCESS',
          rows_read = {rows_read},
          rows_written = {rows_written},
          rows_rejected = {rows_rejected},
          records_before = {records_before},
          records_after = {records_after}
        WHERE run_id = '{run_id}'
    """)


def end_audit_fail(run_id, error_message):
    safe_error = error_message.replace("'", "''")[:5000]

    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'FAILED',
          error_message = '{safe_error}'
        WHERE run_id = '{run_id}'
    """)


def log_rejects(df_rejects, run_id, table_name):
    """Write Silver rejects using the common reject table schema."""
    if df_rejects.isEmpty():
        return

    (
        df_rejects
        .withColumn("reject_id", F.expr("uuid()"))
        .withColumn("run_id", lit(run_id))
        .withColumn("batch_id", lit(batch_id))
        .withColumn("pipeline_name", lit(pipeline_name))
        .withColumn("layer_name", lit(layer_name))
        .withColumn("table_name", lit(table_name))
        .withColumn("source_system", lit(source_system))
        .withColumn("rejected_at", current_timestamp())
        .withColumn("created_at", current_timestamp())
        .select(
            "reject_id", "run_id", "batch_id", "pipeline_name", "layer_name",
            "table_name", "source_system", "source_record_id", "reject_stage",
            "reject_reason", "reject_column", "severity", "raw_payload",
            "error_code", "error_message", "rejected_at", "created_at"
        )
        .write.format("delta")
        .mode("append")
        .saveAsTable(reject_table)
    )

# =========================================================
# LAYER RUN TRACKING
# =========================================================
def get_unprocessed_bronze_run_ids(table_name):
    return [
        r.run_id
        for r in spark.sql(f"""
            SELECT DISTINCT f.run_id
            FROM retail.audit.ingested_files f
            LEFT ANTI JOIN {processed_runs_table} p
              ON f.run_id = p.source_run_id
             AND f.table_name = p.table_name
             AND p.source_layer = 'bronze'
             AND p.target_layer = 'silver'
             AND p.status = 'SUCCESS'
            WHERE f.table_name = '{table_name}'
              AND f.ingestion_status = 'SUCCESS'
        """).collect()
    ]


def log_layer_processed_run(
    source_run_id,
    target_run_id,
    table_name,
    status,
    rows_read,
    rows_written,
    rows_rejected,
    start_time,
    end_time,
    error_message=None
):
    safe_error = "NULL" if error_message is None else "'" + error_message.replace("'", "''")[:5000] + "'"

    spark.sql(f"""
        INSERT INTO {processed_runs_table}
        SELECT
          uuid() AS tracking_id,
          '{source_run_id}' AS source_run_id,
          '{target_run_id}' AS target_run_id,
          '{pipeline_name}' AS pipeline_name,
          'bronze' AS source_layer,
          'silver' AS target_layer,
          '{table_name}' AS table_name,
          '{status}' AS status,
          CAST({rows_read} AS BIGINT) AS rows_read,
          CAST({rows_written} AS BIGINT) AS rows_written,
          CAST({rows_rejected} AS BIGINT) AS rows_rejected,
          timestamp('{start_time}') AS start_time,
          timestamp('{end_time}') AS end_time,
          unix_timestamp(timestamp('{end_time}')) - unix_timestamp(timestamp('{start_time}')) AS duration_seconds,
          {safe_error} AS error_message,
          current_timestamp() AS created_at,
          current_timestamp() AS updated_at
    """)

# =========================================================
# SILVER HELPER FUNCTIONS
# =========================================================

def normalize_string(c):
    return trim(col(c).cast("string"))


def try_cast_sql(c, data_type):
    return expr(f"try_cast(trim(cast(`{c}` as string)) as {data_type})")


def build_typed_expr(c, rule):
    value = trim(col(c).cast("string"))

    if rule == "string":
        return value

    elif rule == "int":
        return try_cast_sql(c, "int")

    elif rule == "int_nullable":
        return when(value == "", None).otherwise(try_cast_sql(c, "int"))

    elif rule == "decimal":
        return try_cast_sql(c, "decimal(18,2)")

    elif rule == "timestamp":
        return try_cast_sql(c, "timestamp")

    elif rule == "date":
        return try_cast_sql(c, "date")

    elif rule == "date_nullable":
        return when(value == "", None).otherwise(try_cast_sql(c, "date"))

    elif rule == "boolean_flag":
        return (
            when(lower(value).isin("true", "1", "yes", "y"), lit(True))
            .when(lower(value).isin("false", "0", "no", "n"), lit(False))
            .otherwise(None)
        )

    else:
        return col(c)


def build_invalid_condition(df, typed_columns, required_columns):
    invalid_condition = None

    # 1. Required column null/blank check
    for c in required_columns:
        cond = col(c).isNull() | (trim(col(c).cast("string")) == "")
        invalid_condition = cond if invalid_condition is None else (invalid_condition | cond)

    # 2. Type cast validation
    for c, rule in typed_columns:
        raw_value = trim(col(c).cast("string"))

        if rule in ["int", "int_nullable"]:
            casted_value = try_cast_sql(c, "int")

            cond = (
                raw_value.isNotNull()
                & (raw_value != "")
                & casted_value.isNull()
            )

        elif rule == "decimal":
            casted_value = try_cast_sql(c, "decimal(18,2)")

            cond = (
                raw_value.isNotNull()
                & (raw_value != "")
                & casted_value.isNull()
            )

        elif rule == "timestamp":
            casted_value = try_cast_sql(c, "timestamp")

            cond = (
                raw_value.isNotNull()
                & (raw_value != "")
                & casted_value.isNull()
            )

        elif rule in ["date", "date_nullable"]:
            casted_value = try_cast_sql(c, "date")

            cond = (
                raw_value.isNotNull()
                & (raw_value != "")
                & casted_value.isNull()
            )

        elif rule == "boolean_flag":
            cond = (
                raw_value.isNotNull()
                & (raw_value != "")
                & (~lower(raw_value).isin("true", "false", "1", "0", "yes", "no", "y", "n"))
            )

        else:
            cond = lit(False)

        invalid_condition = cond if invalid_condition is None else (invalid_condition | cond)

    return invalid_condition

# =========================================================
# MAIN SILVER INCREMENTAL MERGE
# =========================================================
def process_table_incremental_merge(cfg):
    table_name = cfg["table_name"]
    source_table = cfg["source_table"]
    target_table = cfg["target_table"]
    business_key = cfg["business_key"]
    typed_columns = cfg["typed_columns"]
    required_columns = cfg["required_columns"]

    run_id = generate_run_id()
    start_time = datetime.now()

    start_audit(run_id, table_name)

    try:
        records_before = spark.table(target_table).count()

        bronze_run_ids = get_unprocessed_bronze_run_ids(table_name)

        if not bronze_run_ids:
            end_time = datetime.now()

            end_audit_success(
                run_id=run_id,
                rows_read=0,
                rows_written=0,
                rows_rejected=0,
                records_before=records_before,
                records_after=records_before
            )

            print(f"{target_table}: no new Bronze runs to process")
            return

        print(f"{table_name}: processing Bronze run_ids = {bronze_run_ids}")

        # Read only new Bronze runs
        df_bronze = (
            spark.table(source_table)
            .filter(col("etl_run_id").isin(bronze_run_ids))
        )

        rows_read = df_bronze.count()

        if rows_read == 0:
            end_time = datetime.now()

            end_audit_success(
                run_id=run_id,
                rows_read=0,
                rows_written=0,
                rows_rejected=0,
                records_before=records_before,
                records_after=records_before
            )

            print(f"{target_table}: Bronze run IDs found, but no matching rows in Bronze")
            return

        # Validate records
        invalid_condition = build_invalid_condition(
            df_bronze,
            typed_columns,
            required_columns
        )

        df_rejects = (
            df_bronze
            .filter(invalid_condition)
            .withColumn("source_record_id", normalize_string(business_key))
            .withColumn("reject_stage", lit("SILVER"))
            .withColumn("reject_reason", lit("INVALID_CAST_OR_REQUIRED_FIELD"))
            .withColumn("reject_column", lit(",".join(required_columns)))
            .withColumn("severity", lit("ERROR"))
            .withColumn("raw_payload", to_json(struct(*[col(c) for c in df_bronze.columns])))
            .withColumn("error_code", lit("SLV_001"))
            .withColumn("error_message", lit("Required field missing/blank or failed type cast"))
        )

        rows_rejected = df_rejects.count()
        log_rejects(df_rejects, run_id, table_name)

        df_valid = df_bronze.filter(~invalid_condition)

        # Type casting
        typed_select_exprs = [
            build_typed_expr(c, rule).alias(c)
            for c, rule in typed_columns
        ]

        df_silver = df_valid.select(*typed_select_exprs)

        # Deduplicate within incremental batch
        # Silver is maintained as a latest-state trusted table.
        # If multiple versions of the same business key arrive in the same batch,
        # keep the latest version by source etl_loaded_at/created_at.
        window_spec = Window.partitionBy(business_key).orderBy(
            col("etl_loaded_at").desc(),
            col("created_at").desc(),
            col("record_hash").desc()
        )

        df_silver_deduped = (
            df_silver
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(col("rn") == 1)
            .drop("rn")
        )

        target_cols = spark.table(target_table).columns
        df_silver_deduped = df_silver_deduped.select(*target_cols)

        rows_prepared_for_merge = df_silver_deduped.count()

        df_silver_deduped.createOrReplaceTempView(
            f"vw_{table_name}_silver_incremental"
        )

        update_set = ",\n".join([
            f"target.{c} = source.{c}"
            for c in target_cols
            if c != business_key
        ])

        insert_cols = ", ".join(target_cols)
        insert_vals = ", ".join([f"source.{c}" for c in target_cols])

        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING vw_{table_name}_silver_incremental AS source
            ON target.{business_key} = source.{business_key}

            WHEN MATCHED AND target.record_hash <> source.record_hash THEN
              UPDATE SET
                {update_set}

            WHEN NOT MATCHED THEN
              INSERT ({insert_cols})
              VALUES ({insert_vals})
        """

        spark.sql(merge_sql)

        records_after = spark.table(target_table).count()
        end_time = datetime.now()

        end_audit_success(
            run_id=run_id,
            rows_read=rows_read,
            rows_written=rows_prepared_for_merge,
            rows_rejected=rows_rejected,
            records_before=records_before,
            records_after=records_after
        )

        # Mark Bronze runs as processed into Silver
        for bronze_run_id in bronze_run_ids:
            run_df = df_bronze.filter(col("etl_run_id") == bronze_run_id)

            run_rows_read = run_df.count()

            run_reject_condition = build_invalid_condition(
                run_df,
                typed_columns,
                required_columns
            )

            run_rows_rejected = run_df.filter(run_reject_condition).count()

            run_valid_df = run_df.filter(~run_reject_condition)

            run_rows_written = run_valid_df.select(business_key).distinct().count()

            log_layer_processed_run(
                source_run_id=bronze_run_id,
                target_run_id=run_id,
                table_name=table_name,
                status="SUCCESS",
                rows_read=run_rows_read,
                rows_written=run_rows_written,
                rows_rejected=run_rows_rejected,
                start_time=start_time,
                end_time=end_time
            )

        print(f"{target_table}: Silver incremental MERGE completed")
        print(f"Rows read: {rows_read}")
        print(f"Rows rejected: {rows_rejected}")
        print(f"Rows prepared for merge: {rows_prepared_for_merge}")
        print(f"Before: {records_before}")
        print(f"After: {records_after}")

    except Exception as e:
        end_time = datetime.now()
        error_message = str(e)

        end_audit_fail(run_id, error_message)

        print(f"{target_table}: FAILED")
        print(error_message)

        raise


for cfg in table_configs:
    process_table_incremental_merge(cfg)